# Flight arrival delay prediction

In [ ]:
import gc
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

csv_path = "flight_data_2024.csv"
artifact_path = "arrival_delay_model.pt"
target_col = "arr_delay"

observed_dep_delay = True

val_fraction = 0.15
test_fraction = 0.15

batch_size = 4096
eval_batch_size = 16384
max_epochs = 25
warmup_epochs = 2
min_lr = 1e-5
patience = 10
lr = 3e-3
weight_decay = 1e-3
grad_clip = 5.0
ema_decay = 0.8
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1. Load data

In [ ]:
dtypes = {
    "month": "int16",
    "day_of_month": "int16",
    "day_of_week": "int16",
    "crs_dep_time": "int32",
    "crs_arr_time": "int32",
    "dep_delay": "float32",
    "arr_delay": "float32",
    "cancelled": "int8",
    "diverted": "int8",
    "crs_elapsed_time": "float32",
    "distance": "float32",
}

df = pd.read_csv(csv_path, usecols=list(dtypes), dtype=dtypes)
df.shape

In [ ]:
df.head()

In [ ]:
print(df.isna().sum()[lambda counts: counts > 0])
df.describe().T

## 2. Eda

In [ ]:
operating = df[(df.cancelled == 0) & (df.diverted == 0)]
operating[target_col].describe()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(operating[target_col].clip(-60, 180), bins=100)
plt.axvline(0, color="red", linestyle="--")
plt.title("arrival delay distribution, axis clipped for display")
plt.xlabel("minutes, negative is early")
plt.show()

In [ ]:
operating.groupby("month")[target_col].mean().plot(marker="o", figsize=(8, 3))
plt.title("mean arrival delay by month")
plt.show()

In [ ]:
operating.groupby("day_of_week")[target_col].mean().plot(marker="o", figsize=(8, 3))
plt.title("mean arrival delay by day of week")
plt.show()

## 3. Clean

In [ ]:
observed_cols = ["dep_delay"] if observed_dep_delay else []
required_cols = [target_col, "crs_elapsed_time", "crs_dep_time", "crs_arr_time", "distance", *observed_cols]

before = len(df)
df = df[(df.cancelled == 0) & (df.diverted == 0)]
df = df.dropna(subset=required_cols).reset_index(drop=True)
print(f"kept {len(df)} of {before} rows")

## 4. Split

In [ ]:
day_key = df.month.to_numpy() * 32 + df.day_of_month.to_numpy()
days = np.unique(day_key)

n_test = max(1, round(len(days) * test_fraction))
n_val = max(1, round(len(days) * val_fraction))
test_days = days[-n_test:]
val_days = days[-n_test - n_val:-n_test]
train_days = days[:-n_test - n_val]

rows_train = np.flatnonzero(np.isin(day_key, train_days))
rows_val = np.flatnonzero(np.isin(day_key, val_days))
rows_test = np.flatnonzero(np.isin(day_key, test_days))

for name, idx in [("train", rows_train), ("val", rows_val), ("test", rows_test)]:
    keys = day_key[idx]
    first, last = keys.min(), keys.max()
    print(f"{name:5s} {len(idx):>8d} rows  {first // 32:02d}-{first % 32:02d} to {last // 32:02d}-{last % 32:02d}")

## 5. Features

Every column is a field airlabs returns. No airline, airport or carrier codes, because the training data is 15 us carriers and 348 us airports and those codes would mean nothing for the rest of the world. The bts `weather_delay` and `nas_delay` columns are left out too, since they record the cause of a delay after the fact.

In [ ]:
def hhmm_to_mins(values):
    return ((values // 100) % 24) * 60 + (values % 100)


dep_mins = hhmm_to_mins(df.crs_dep_time.to_numpy()).astype(np.float32)
arr_mins = hhmm_to_mins(df.crs_arr_time.to_numpy()).astype(np.float32)

num_values = {
    "crs_elapsed_time": df.crs_elapsed_time.to_numpy(dtype=np.float32),
    "distance": df.distance.to_numpy(dtype=np.float32),
    "day_of_month": df.day_of_month.to_numpy(dtype=np.float32),
    "dep_sin": np.sin(2 * np.pi * dep_mins / 1440).astype(np.float32),
    "dep_cos": np.cos(2 * np.pi * dep_mins / 1440).astype(np.float32),
    "arr_sin": np.sin(2 * np.pi * arr_mins / 1440).astype(np.float32),
    "arr_cos": np.cos(2 * np.pi * arr_mins / 1440).astype(np.float32),
}
if observed_dep_delay:
    num_values["dep_delay"] = df.dep_delay.to_numpy(dtype=np.float32)

num_cols = list(num_values)
n_months = 12
n_dow = 7
n_features = len(num_cols) + n_months + n_dow

X = np.empty((len(df), n_features), dtype=np.float32)
all_rows = np.arange(len(df))

for column, name in enumerate(num_cols):
    X[:, column] = num_values[name]

offset = len(num_cols)
X[all_rows, offset + df.month.to_numpy() - 1] = 1.0
X[all_rows, offset + n_months + df.day_of_week.to_numpy() - 1] = 1.0

assert np.isfinite(X).all()

y = df[target_col].to_numpy(dtype=np.float32).copy()
del df, num_values, all_rows
gc.collect()

X.shape, y.shape

## 6. Scale

In [ ]:
def fit_scaler(index, block=500_000):
    total = np.zeros(n_features, dtype=np.float64)
    total_sq = np.zeros(n_features, dtype=np.float64)
    for start in range(0, len(index), block):
        chunk = X[index[start:start + block]].astype(np.float64)
        total += chunk.sum(0)
        chunk **= 2
        total_sq += chunk.sum(0)
    mean = total / len(index)
    scale = np.sqrt(np.maximum(total_sq / len(index) - mean ** 2, 0.0))
    scale[scale < 1e-6] = 1.0
    return mean.astype(np.float32), scale.astype(np.float32)


x_mean, x_scale = fit_scaler(rows_train)
X -= x_mean
X /= x_scale
gc.collect()

X[:2, :6]

## 7. Tensors

In [ ]:
def pick_storage(needed):
    if device.type != "cuda":
        return torch.device("cpu")
    free, _ = torch.cuda.mem_get_info()
    return device if free * 0.6 > needed else torch.device("cpu")


storage = pick_storage(X.nbytes + y.nbytes + 64 * 1024 ** 2)
features = torch.from_numpy(X).to(storage)
labels = torch.from_numpy(y).to(storage)
del X, y
gc.collect()

split_rows = {
    name: torch.from_numpy(index).to(storage)
    for name, index in [("train", rows_train), ("val", rows_val), ("test", rows_test)]
}

features.shape, labels.shape, storage

In [ ]:
generator = torch.Generator(device=split_rows["train"].device)
generator.manual_seed(seed)


def iterate(index, size, shuffle=False, gen=None):
    if shuffle:
        index = index[torch.randperm(index.numel(), device=index.device, generator=gen)]
    for start in range(0, index.numel(), size):
        batch = index[start:start + size]
        yield features[batch], labels[batch].unsqueeze(1)


def train_batches():
    return iterate(split_rows["train"], batch_size, shuffle=True, gen=generator)


def eval_batches(split):
    return iterate(split_rows[split], eval_batch_size)


next(train_batches())[0].shape

## 8. Model

In [ ]:
class DelayPredictor(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


model = DelayPredictor(features.shape[1]).to(device)
model

## 9. Train

In [ ]:
steps = -(-split_rows["train"].numel() // batch_size)
total_steps = max_epochs * steps
warmup_steps = warmup_epochs * steps


def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = min((step - warmup_steps) / max(1, total_steps - warmup_steps), 1.0)
    floor = min_lr / lr
    return floor + (1.0 - floor) * 0.5 * (1.0 + math.cos(math.pi * progress))


criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val = float("inf")
smoothed = None
best_state = None
stale = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    loss_sum = torch.zeros((), device=device)
    seen = 0
    for xb, yb in train_batches():
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()
        loss_sum += loss.detach() * yb.size(0)
        seen += yb.size(0)
    train_loss = loss_sum.item() / seen

    model.eval()
    error_sum = torch.zeros((), device=device)
    seen = 0
    with torch.no_grad():
        for xb, yb in eval_batches("val"):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            error_sum += (model(xb).squeeze(1) - yb.squeeze(1)).abs().sum()
            seen += yb.size(0)
    val_mae = error_sum.item() / seen
    smoothed = val_mae if smoothed is None else ema_decay * smoothed + (1 - ema_decay) * val_mae
    print(f"epoch {epoch:02d}  train {train_loss:8.4f}  val_mae {val_mae:7.4f}  ema {smoothed:7.4f}  lr {optimizer.param_groups[0]['lr']:.2e}")

    if smoothed < best_val - 1e-4:
        best_val = smoothed
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= patience:
            print(f"early stop at epoch {epoch}, best ema val_mae {best_val:.4f}")
            break

model.load_state_dict(best_state)
model.eval()

## 10. Evaluate

In [ ]:
@torch.no_grad()
def predict(split):
    model.eval()
    parts = [model(xb.to(device, non_blocking=True)).squeeze(1).cpu() for xb, _ in eval_batches(split)]
    return torch.cat(parts).numpy()

In [ ]:
preds = predict("test")
actual = labels[split_rows["test"]].cpu().numpy()
baseline = np.full_like(actual, labels[split_rows["train"]].mean().item())

errors = np.abs(actual - preds)
metrics = {
    "mae": mean_absolute_error(actual, preds),
    "rmse": float(np.sqrt(mean_squared_error(actual, preds))),
    "r2": r2_score(actual, preds),
    "median ae": float(np.median(errors)),
    "p90 ae": float(np.percentile(errors, 90)),
    "within 15": float((errors <= 15).mean()),
    "within 30": float((errors <= 30).mean()),
    "bias": float((preds - actual).mean()),
    "baseline mae": mean_absolute_error(actual, baseline),
    "baseline rmse": float(np.sqrt(mean_squared_error(actual, baseline))),
}

print(f"best ema val_mae {best_val:.4f}")
print(f"target mean {actual.mean():8.3f}  std {actual.std():8.3f}")
for name, value in metrics.items():
    print(f"{name:14s} {value:9.4f}")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(actual[:20000], preds[:20000], s=2, alpha=0.15)
axes[0].plot([-60, 180], [-60, 180], "r--")
axes[0].set_xlabel("actual arrival delay, min")
axes[0].set_ylabel("predicted arrival delay, min")
axes[0].set_title("test set, predicted vs actual")

axes[1].hist(preds - actual, bins=100)
axes[1].set_xlabel("prediction error, min")
axes[1].set_title(f"error distribution, mae {metrics['mae']:.2f} min")

plt.tight_layout()
plt.show()

## 11. Save

In [ ]:
torch.save(
    {
        "state_dict": best_state,
        "config": {
            "target": target_col,
            "numeric_columns": num_cols,
            "month_levels": list(range(1, n_months + 1)),
            "dow_levels": list(range(1, n_dow + 1)),
            "observed_departure_delay": observed_dep_delay,
            "hidden": [256, 128, 64],
            "dropout": 0.2,
            "split_days": {
                "train": train_days.tolist(),
                "val": val_days.tolist(),
                "test": test_days.tolist(),
            },
        },
        "x_mean": torch.from_numpy(x_mean),
        "x_scale": torch.from_numpy(x_scale),
        "metrics": {
            "val_mae": best_val,
            "test_mae": metrics["mae"],
            "test_rmse": metrics["rmse"],
            "test_r2": metrics["r2"],
        },
    },
    artifact_path,
)

print(f"saved {artifact_path}")
print(metrics)